# 01 — FastAPI complet

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- créer une API REST avec FastAPI et la servir avec `uvicorn`;
- définir des **schemas Pydantic** pour les entrées/sorties;
- utiliser les **path/query/body parameters** typés;
- organiser le code en **routers** et utiliser la **dependency injection**;
- générer automatiquement la **documentation OpenAPI** (Swagger UI, ReDoc);
- gérer erreurs, middlewares, CORS, auth (bearer);
- tester l'API avec `httpx.TestClient` / `TestClient` FastAPI;


## Prérequis — ce que vous devez maîtriser

Ce notebook fait partie des **Side tracks**. Il est conçu pour être autonome, à condition d'avoir suivi le niveau **Intermédiaire** de la formation principale. Vous devez donc être à l'aise avec :

- Python moderne : type hints, dataclasses, async/await;
- Pydantic v2 (modèles, validation) — vu en Intermédiaire;
- HTTP de base : méthodes, codes de statut, JSON;
- un peu de SQLAlchemy si vous voulez persister (pas obligatoire);

Ce que ce notebook **ne couvrira pas** (ou seulement en mention) :

- les déploiements en production (Docker, k8s) — un notebook dédié viendra;
- les WebSockets (voir `05_Websockets_realtime.ipynb`);
- l'ORM complet — voir `03_SQLAlchemy_avance.ipynb`;


> **Note.** FastAPI est **async-first**, s'appuie sur les **type hints** pour la validation et la sérialisation, et génère automatiquement un schéma OpenAPI. Il est devenu le standard moderne pour les APIs REST en Python.


## Plan

1. Installation et premier 'Hello'
2. Lancer avec uvicorn
3. Path parameters typés
4. Query parameters
5. Body : Pydantic models
6. Réponses et `response_model`
7. Codes de statut et `HTTPException`
8. Routers
9. Dependency Injection
10. Middlewares et CORS
11. Auth : Bearer token
12. Requêtes à la base (SQLAlchemy async)
13. Documentation OpenAPI
14. Tests avec `TestClient`
15. Structure d'un projet FastAPI
16. Synthèse
17. Exercices


---


## 1. Installation et premier Hello

```bash
pip install 'fastapi[standard]'
```

Ce méta-paquet installe `fastapi`, `uvicorn`, et `httpx` pour les tests.


In [ ]:
# app.py
from fastapi import FastAPI

app = FastAPI()

@app.get('/')
def racine() -> dict[str, str]:
    return {'message': 'Bonjour FastAPI'}

---


## 2. Lancer avec uvicorn

```bash
uvicorn app:app --reload --host 0.0.0.0 --port 8000
```

- `app:app` = module:instance
- `--reload` = rechargement automatique en dev

L'API est disponible sur `http://localhost:8000/`, la doc interactive sur `/docs` (Swagger) et `/redoc`.


---


## 3. Path parameters typés


In [ ]:
from fastapi import FastAPI
app = FastAPI()

@app.get('/utilisateurs/{user_id}')
def lire_utilisateur(user_id: int) -> dict:
    return {'id': user_id, 'nom': 'Alice'}

FastAPI convertit automatiquement `user_id` en `int`. Si le client envoie `/utilisateurs/abc`, FastAPI répond avec une erreur 422 claire et générée.


---


## 4. Query parameters


In [ ]:
from fastapi import FastAPI, Query
app = FastAPI()

@app.get('/recherche')
def rechercher(
    q: str = Query(..., min_length=2, max_length=50),
    page: int = 1,
    taille: int = Query(default=10, le=100),
) -> dict:
    return {'q': q, 'page': page, 'taille': taille}

**Astuce** : tout argument sans valeur par défaut devient obligatoire. Les contraintes passent par `Query(..., gt=0, max_length=50)`.


---


## 5. Body : Pydantic models


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI()

class CreerUtilisateur(BaseModel):
    nom: str = Field(min_length=1, max_length=100)
    email: str
    age: int = Field(ge=0, le=150)

@app.post('/utilisateurs')
def creer(payload: CreerUtilisateur) -> dict:
    return {'ok': True, 'nom': payload.nom}

FastAPI **valide** le body JSON, lève 422 si le schéma n'est pas respecté, et montre les champs invalides dans Swagger.


---


## 6. Réponses et `response_model`

Utiliser `response_model` filtre automatiquement la réponse pour ne garder que les champs publics et valide leur forme. Utile pour masquer un `password_hash`.


In [ ]:
class UtilisateurPublic(BaseModel):
    id: int
    nom: str
    email: str

@app.get('/utilisateurs/{uid}', response_model=UtilisateurPublic)
def lire_public(uid: int) -> dict:
    return {'id': uid, 'nom': 'Alice', 'email': 'a@x.com', 'password_hash': 'secret'}

Le champ `password_hash` ne sera **pas** renvoyé car absent du `response_model`.


---


## 7. Codes de statut et `HTTPException`


In [ ]:
from fastapi import FastAPI, HTTPException, status

app = FastAPI()
BASE = {1: 'Alice'}

@app.get('/u/{uid}', status_code=200)
def lire(uid: int) -> dict:
    if uid not in BASE:
        raise HTTPException(status_code=status.HTTP_404_NOT_FOUND, detail='non trouvé')
    return {'id': uid, 'nom': BASE[uid]}

In [ ]:
@app.post('/u', status_code=status.HTTP_201_CREATED)
def creer(n: str) -> dict:
    return {'nom': n}

---


## 8. Routers — organiser les endpoints


In [ ]:
# users.py
from fastapi import APIRouter

router = APIRouter(prefix='/users', tags=['users'])

@router.get('/')
def liste() -> list[str]:
    return ['Alice', 'Bob']

In [ ]:
# main.py
from fastapi import FastAPI
# from users import router as users_router

app = FastAPI()
# app.include_router(users_router)

Les routers permettent de découper l'API par domaine fonctionnel et d'appliquer des tags communs (qui regroupent dans Swagger).


---


## 9. Dependency Injection

Le système de DI de FastAPI permet d'injecter une ressource (DB, config, utilisateur courant) dans n'importe quel endpoint.


In [ ]:
from fastapi import FastAPI, Depends

app = FastAPI()

def pagination(page: int = 1, taille: int = 10) -> dict:
    return {'page': page, 'taille': taille}

@app.get('/posts')
def posts(p: dict = Depends(pagination)) -> dict:
    return p

**Chaque dépendance peut elle-même avoir des dépendances**. On compose des arbres d'injection complexes sans frameworks d'IoC externes.


---


## 10. Middlewares et CORS


In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=['https://monsite.fr'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

---


## 11. Auth : Bearer token


In [ ]:
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials

app = FastAPI()
security = HTTPBearer()

def verifier_token(creds: HTTPAuthorizationCredentials = Depends(security)) -> str:
    if creds.credentials != 'secret-demo':
        raise HTTPException(status_code=401, detail='token invalide')
    return 'alice'

@app.get('/moi')
def moi(user: str = Depends(verifier_token)) -> dict:
    return {'user': user}

---


## 12. SQLAlchemy async — l'exemple minimal


In [ ]:
# Dépendance qui ouvre une session async
# from sqlalchemy.ext.asyncio import AsyncSession, create_async_engine, async_sessionmaker
# engine = create_async_engine('sqlite+aiosqlite:///app.db', echo=False)
# SessionLocal = async_sessionmaker(engine, expire_on_commit=False)

# async def get_session():
#     async with SessionLocal() as s:
#         yield s

Le pattern : une dépendance `async def get_session()` qui `yield` la session, et FastAPI s'occupe de fermer proprement après la requête.


---


## 13. Documentation OpenAPI

FastAPI génère automatiquement un schéma OpenAPI 3.1 complet à partir de vos types et Pydantic models. Interfaces prêtes à l'emploi :

- `/docs` — Swagger UI interactif
- `/redoc` — ReDoc (plus lisible pour la doc)
- `/openapi.json` — le schéma JSON brut

Vous pouvez personnaliser titre, version, description :


In [ ]:
app = FastAPI(
    title='Mon API',
    version='1.0.0',
    description='API de démonstration',
    contact={'name': 'Équipe', 'email': 'team@example.com'},
)

---


## 14. Tests avec `TestClient`


In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()

@app.get('/ping')
def ping() -> dict: return {'pong': True}

client = TestClient(app)
resp = client.get('/ping')
print(resp.status_code, resp.json())

`TestClient` utilise `httpx` sous le capot et lance l'app **en mémoire** sans démarrer uvicorn. Idéal pour les tests unitaires et CI.


---


## 15. Structure d'un projet FastAPI


```
mon_api/
├── src/
│   └── mon_api/
│       ├── __init__.py
│       ├── main.py          # FastAPI() + include_router
│       ├── api/
│       │   ├── deps.py      # dépendances communes
│       │   ├── users.py     # router users
│       │   └── items.py
│       ├── core/config.py   # Settings (Pydantic BaseSettings)
│       ├── db/models.py
│       └── schemas/
│           └── user.py      # Pydantic models
├── tests/
└── pyproject.toml
```


---


## Synthèse

| Notion | À retenir |
|---|---|
| `FastAPI()` | instance, appelée par uvicorn |
| Décorateurs | `@app.get`, `@app.post`, `@app.put`, `@app.delete`, `@app.patch` |
| Path parameters | typés via les annotations (`user_id: int`) |
| Query parameters | arguments sans annotation path → query |
| Body | Pydantic `BaseModel` dans la signature |
| `response_model` | filtre la sortie, masque les champs privés |
| `Depends` | dependency injection native |
| `HTTPException` | lever une erreur HTTP typée |
| Routers | organiser par domaine, `prefix`, `tags` |
| `TestClient` | tests en mémoire via httpx |


### Règles à retenir

- **Toujours typer** vos signatures — FastAPI l'exploite pour validation et doc.
- **Pydantic pour les bodies**, jamais `dict` — pour avoir la validation.
- **`response_model`** pour filtrer les réponses et protéger les champs privés.
- **CORS explicite** en production — jamais `allow_origins=['*']` avec credentials.
- **`async def`** pour les endpoints I/O-bound, `def` pour les purement CPU.


---


## Exercices

Pour chaque exercice : écrivez une **fonction typée**, testez-la, puis comparez avec la correction. N'ouvrez pas la correction avant d'avoir vraiment cherché.


### Exercice 1 — Endpoint qui double un nombre

Écrivez une app FastAPI avec `GET /double/{n}` renvoyant `{'resultat': 2 * n}`.


In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.get('/double/{n}')
def double(n: int) -> dict:
    ...


<details>
<summary>📖 Voir la correction</summary>

```python
from fastapi import FastAPI

app = FastAPI()

@app.get('/double/{n}')
def double(n: int) -> dict:
    return {'resultat': 2 * n}
```

</details>


### Exercice 2 — Création d'article

Écrivez `POST /articles` qui accepte un JSON `{titre, contenu}` (Pydantic), renvoie le statut 201 et le payload reçu.


In [ ]:
from fastapi import FastAPI, status
from pydantic import BaseModel

app = FastAPI()

class Article(BaseModel):
    titre: str
    contenu: str

@app.post('/articles', status_code=status.HTTP_201_CREATED)
def creer(a: Article) -> Article:
    ...


<details>
<summary>📖 Voir la correction</summary>

```python
from fastapi import FastAPI, status
from pydantic import BaseModel

app = FastAPI()

class Article(BaseModel):
    titre: str
    contenu: str

@app.post('/articles', status_code=status.HTTP_201_CREATED)
def creer(a: Article) -> Article:
    return a
```

</details>


### Exercice 3 — Endpoint avec dépendance

Ajoutez une dépendance `verifier_api_key(x_api_key: str = Header(...))` qui lève 401 si la clé n'est pas `'demo'`. Protégez `GET /prive`.


In [ ]:
from fastapi import FastAPI, Header, HTTPException, Depends

app = FastAPI()

def verifier_api_key(x_api_key: str = Header(...)) -> None:
    ...

@app.get('/prive', dependencies=[Depends(verifier_api_key)])
def prive() -> dict:
    return {'ok': True}


<details>
<summary>📖 Voir la correction</summary>

```python
from fastapi import FastAPI, Header, HTTPException, Depends

app = FastAPI()

def verifier_api_key(x_api_key: str = Header(...)) -> None:
    if x_api_key != 'demo':
        raise HTTPException(status_code=401, detail='clé invalide')

@app.get('/prive', dependencies=[Depends(verifier_api_key)])
def prive() -> dict:
    return {'ok': True}
```

</details>


### Exercice 4 — Test d'un endpoint

Écrivez un test avec `TestClient` qui vérifie que `GET /` renvoie un JSON contenant la clé `message`.


In [ ]:
from fastapi.testclient import TestClient
# app = ...

def test_racine() -> None:
    ...


<details>
<summary>📖 Voir la correction</summary>

```python
from fastapi.testclient import TestClient
from fastapi import FastAPI

app = FastAPI()
@app.get('/')
def r() -> dict: return {'message': 'hi'}

def test_racine() -> None:
    client = TestClient(app)
    resp = client.get('/')
    assert resp.status_code == 200
    assert 'message' in resp.json()
```

</details>


---


## Ressources externes

- [Documentation FastAPI](https://fastapi.tiangolo.com/)
- [FastAPI full stack template](https://github.com/fastapi/full-stack-fastapi-template)
- [Pydantic v2](https://docs.pydantic.dev/latest/)
- [Starlette (base)](https://www.starlette.io/)
